# Data Loading

## 1. dataset

In [33]:
from datasets import load_dataset
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq


### Connect to Educational fineweb dataset

In [34]:

# 1 Connect
fineweb = load_dataset(
    "HuggingFaceFW/fineweb-edu",
    name="sample-10BT",
    split="train",
    streaming=True,
)

    # add to our corpus

### Get a sample

In [43]:
fineweb_sample = []

for row in fineweb:
    
    fineweb_sample.append(row)
    
    if len(fineweb_sample) >= 150:
        break

print(f"Loaded {len(fineweb_sample):,} documents")

Loaded 150 documents


### Calculate dataset statistics 

In [45]:
# Cell 5 — Calculate dataset statistics

num_documents = 0
total_characters = 0
min_length = None
max_length = 0

for row in fineweb_sample:
    
    text = row["text"]

    length = len(text)

    num_documents += 1
    total_characters += length

    if min_length is None:
        min_length = length
    else:
        min_length = min(min_length, length)

    max_length = max(max_length, length)

print(f"Documents:          {num_documents:,}")
print(f"Characters/tokens:  {total_characters:,}")
print(f"Average length:     {total_characters / num_documents:,.0f}")
print(f"Minimum length:     {min_length:,}")
print(f"Maximum length:     {max_length:,}")

Documents:          150
Characters/tokens:  533,917
Average length:     3,559
Minimum length:     335
Maximum length:     18,639


### Compare corpus lengths

In [51]:
# Shakespear
sp_corpus = "../data/tiny_shakespear.txt"
sp_data_path = Path(sp_corpus)
sp_text = sp_data_path.read_text(encoding="utf-8",)

sp_text_length = len(sp_text)
print(f"Shakespeare text_length : {sp_text_length}")

# Tiny Stories
ts_corpus = "../data/tinystories_300k.txt"
ts_data_path = Path(ts_corpus)
ts_text = ts_data_path.read_text(encoding="utf-8",)

ts_text_length = len(ts_text)
print(f"Tiny Stories text_length : {ts_text_length}")


Shakespeare text_length : 1115394
Tiny Stories text_length : 299987


In [55]:
#print(repr(sp_text[:2_000]))
#print(repr(ts_text[:2_000]))
for marker in ["\nOnce upon a time", "\nOne day"]:
    print(repr(marker), ts_text.count(marker))

'\nOnce upon a time' 221
'\nOne day' 107


### Save to parquet

In [47]:
output_path = Path("../data/fineweb-edu-sample.parquet")
output_path.parent.mkdir(parents=True, exist_ok=True)

table = pa.Table.from_pylist(fineweb_sample)

pq.write_table(
    table,
    output_path,
    )

print(f"Saved {len(fineweb_sample):,} documents")
print(f"Path: {output_path}")

Saved 150 documents
Path: ../data/fineweb-edu-sample.parquet


## Read from parquet

In [50]:
import pyarrow.parquet as pq

table = pq.read_table("../data/fineweb-edu-sample.parquet")

print(f"Rows: {table.num_rows:,}")
print(f"Columns: {table.column_names}")
print(table.schema)

print("=" * 80)

texts = table["text"].to_pylist()

print(f"Loaded {len(texts):,} documents")
print(texts[0][:500])

Rows: 150
Columns: ['text', 'id', 'dump', 'url', 'file_path', 'language', 'language_score', 'token_count', 'score', 'int_score']
text: string
id: string
dump: string
url: string
file_path: string
language: string
language_score: double
token_count: int64
score: double
int_score: int64
Loaded 150 documents
The Independent Jane
For all the love, romance and scandal in Jane Austen’s books, what they are really about is freedom and independence. Independence of thought and the freedom to choose.
Elizabeth’s refusal of Mr. Collins offer of marriage showed an independence seldom seen in heroines of the day. Her refusal of Mr. Darcy while triggered by anger showed a level of independence that left him shocked and stunned.
The freedom she exhibited in finally accepting him in direct defiance of Lady Cath


## Join em

In [ ]:
# from pathlib import Path
# import pandas as pd

# DATA_DIR = Path("data")

# rows = []

# # 1. Load Parquet files
# for path in DATA_DIR.glob("*.parquet"):
#     df = pd.read_parquet(path)

#     # Adapt this to your actual column names
#     if "text" in df.columns:
#         for text in df["text"].dropna():
#             rows.append({
#                 "text": str(text),
#                 "source": path.name,
#             })

# # 2. Load TXT files
# for path in DATA_DIR.glob("*.txt"):
#     text = path.read_text(encoding="utf-8", errors="replace").strip()

#     if text:
#         rows.append({
#             "text": text,
#             "source": path.name,
#         })

# # 3. Create unified dataset
# dataset = pd.DataFrame(rows)

# # 4. Basic cleaning
# dataset["text"] = (
#     dataset["text"]
#     .str.replace(r"\s+", " ", regex=True)
#     .str.strip()
# )

# dataset = dataset[dataset["text"].str.len() > 0]

# # 5. Remove exact duplicates
# dataset = dataset.drop_duplicates(subset=["text"])

# # 6. Shuffle
# dataset = dataset.sample(frac=1, random_state=42).reset_index(drop=True)

# # 7. Save
# dataset.to_parquet("training_dataset.parquet", index=False)

# print(dataset.shape)
# print(dataset.head())

In [ ]:
# from datasets import load_dataset

# tiny_stories_ds = load_dataset("roneneldan/TinyStories", split="train", streaming=True,)

# # for story in tiny_stories_ds.take(3):
# #     print("=" * 80)
# #     print(story["text"])
    
# TARGET = 300_000
# chunks = []
# n = 0

# for row in ds:
#     story = row["text"].strip() + "\n"
    
#     if n + len(story) > TARGET:
#         break
    
#     chunks.append(story)
#     n += len(story)

# with open("../data/tinystories_300k.txt", "w", encoding="utf-8") as f:
#     f.write("".join(chunks))

# print(f"Characters: {n:,}")
# print(f"Stories: {len(chunks):,}")

In [ ]:
## JSONL

## Parquet

In [ ]:
from datatrove.pipeline.readers import ParquetReader

# limit determines how many documents will be streamed (remove for all)
data_reader = ParquetReader("hf://datasets/HuggingFaceFW/fineweb-edu", glob_pattern="data/*/*.parquet", limit=1000)
# or to fetch a specific dump CC-MAIN-2024-10,  eplace "CC-MAIN-2024-10" with "sample/100BT" to use the 100BT sample
data_reader = ParquetReader("hf://datasets/HuggingFaceFW/fineweb-edu/CC-MAIN-2024-10", limit=1000) 
for document in data_reader():
    # do something with document
    print(document)

###############################    
# OR for a processing pipeline:
###############################

from datatrove.executor import LocalPipelineExecutor
from datatrove.pipeline.readers import ParquetReader
from datatrove.pipeline.filters import LambdaFilter
from datatrove.pipeline.writers import JsonlWriter

pipeline_exec = LocalPipelineExecutor(
    pipeline=[
        # replace "CC-MAIN-2024-10" with "sample/100BT" to use the 100BT sample
        ParquetReader("hf://datasets/HuggingFaceFW/fineweb-edu/CC-MAIN-2024-10", limit=1000),
        LambdaFilter(lambda doc: "hugging" in doc.text),
        JsonlWriter("some-output-path")
    ],
    tasks=10
)
pipeline_exec.run()
